# 1D-CNN Localization: PyTorch Exploration Notebook

Explores a **1D Convolutional Neural Network** as an alternative to XGBoost and Random Forest for CSI-based UE localization on the 25x25 grid (4 m spacing).

**Key design decisions:**
- Predicts Cartesian coordinates directly — **regression**, not classification
- Input reshaped to `[N, h+1, n_signals]` — CNN operates along the **time axis**
- Static device profile (n_antennas, gain, UE height) via a separate FC branch
- **L1 loss** (MAE — directly optimizes the positioning evaluation metric)
- Adam + Linear Warmup + Cosine Annealing + gradient clipping + early stopping

**Baselines to beat:**
- XGBoost (absolute h=3): **7.959 m MAE**
- RF (hybrid h=1):        **7.506 m MAE**

> **Why NOT cross-entropy?** Cross-entropy measures divergence between probability distributions over *discrete* classes and is only valid for classification tasks. Since we predict *continuous* (x, y, z) coordinates, we need a regression loss (L1/MAE or Huber).

In [ ]:
import sys, os, time, warnings, json, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Adjust if running from a different directory
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR
for _ in range(10):
    if (PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25').exists():
        break
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / 'experiments' / '09_grid_localization' / 'src' / 'python'))
from utils.read_jsonc import read_jsonc

import datetime
RUN_TIMESTAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
DATA_DIR = PROJECT_ROOT / 'results' / 'grid_localization' / 'grid_25x25' / 'sim_data_ne_bs_2026-06-13_14-54-18'
OUT_DIR  = PROJECT_ROOT / 'results' / 'notebook_experiments' / 'cnn' / f'cnn_exploration_{RUN_TIMESTAMP}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data exists:  {DATA_DIR.exists()}')
print(f'Output dir:   {OUT_DIR}')

## Hyperparameters

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
H                    = 10     # History depth (h+1 timesteps total)
SUBSAMPLE            = 1.00   # Fraction of training data (1.0 = full dataset)
TEST_RATIO           = 0.20   # Chronological split

# ──────────────────────────────────────────────────────────────────────────
BATCH_SIZE           = 256
LR                   = 1e-3   # Peak LR (reached after warmup)
LR_MIN               = 1e-6   # Starting warmup LR and cosine annealing floor
WARMUP_EPOCHS        = 5      # Linear warmup: LR_MIN -> LR
WEIGHT_DECAY         = 1e-4
MAX_EPOCHS           = 100
EARLY_STOP_PATIENCE  = 20     # Generous patience for cosine schedule
GRAD_CLIP            = 1.0

# ──────────────────────────────────────────────────────────────────────────
CONV_CHANNELS        = [64, 128, 256]
DENSE_UNITS          = [512, 256, 128]
DROPOUT_RATE         = 0.25
LOSS_FN              = 'l1'    # 'l1' (MAE), 'huber', or 'mse'
HUBER_DELTA          = 5.0     # meters (only for LOSS_FN='huber')

# ── Columns ────────────────────────────────────────────────────────────
SIGNAL_COLS  = ['rss', 'sinr', 'aoa_azimuth', 'aoa_elevation']
STATIC_COLS  = ['n_antennas', 'antenna_gain_db', 'ue_height']
TARGET_COLS  = ['target_x', 'target_y', 'target_z']
GRID_SPACING = 4.0  # meters

## Data Loading

In [ ]:
from pipelines.multi_user_pipeline_regression import load_all_users

t0 = time.time()
df_raw = load_all_users(DATA_DIR)
print(f'Done in {time.time()-t0:.1f}s  —  {len(df_raw):,} total steps')
df_raw.head(2)

## Target Engineering & Train/Test Split

In [ ]:
from pipelines.multi_user_pipeline_regression import make_split, _read_bs_position_3d

bs_pos = np.array(_read_bs_position_3d(DATA_DIR))
print(f'BS position: {bs_pos} m')

ue_xyz = np.column_stack([df_raw['x_pos'], df_raw['y_pos'], df_raw['ue_height']])
delta  = ue_xyz - bs_pos
df_raw['target_x'], df_raw['target_y'], df_raw['target_z'] = delta[:,0], delta[:,1], delta[:,2]

# Chronological split per user
df_raw = make_split(df_raw, TEST_RATIO)

# Subsample training data (SUBSAMPLE=1.0 uses all data)
train_idxs, test_idxs = [], df_raw[df_raw['split']=='test'].index.tolist()
for uid in df_raw['user_id'].unique():
    utr = df_raw[(df_raw['user_id']==uid) & (df_raw['split']=='train')].sort_values('step_index')
    train_idxs.extend(utr.index[:int(len(utr)*SUBSAMPLE)].tolist())

df_train = df_raw.loc[train_idxs].copy()
df_test  = df_raw.loc[test_idxs].copy()
print(f'Train: {len(df_train):,}  |  Test: {len(df_test):,}')

## Sequence Builder

Reshapes flat per-step data into `[N, h+1, n_signals]` temporal tensors.
The CNN slides its kernel along the **time axis** (dimension 1 before transpose),
learning which local temporal patterns are most informative for localization.

In [ ]:
def build_sequences(df, h):
    all_seq, all_static, all_targets = [], [], []
    for uid in sorted(df['user_id'].unique()):
        udf     = df[df['user_id']==uid].sort_values('step_index').reset_index(drop=True)
        sigs    = udf[SIGNAL_COLS].values.astype(np.float32)
        statics = udf[STATIC_COLS].values.astype(np.float32)
        targets = udf[TARGET_COLS].values.astype(np.float32)
        for i in range(h, len(udf)):
            all_seq.append(sigs[i-h : i+1])  # chronological [h+1, n_signals]
            all_static.append(statics[i])
            all_targets.append(targets[i])
    return np.stack(all_seq), np.stack(all_static), np.stack(all_targets)


print('Building sequences...')
t0 = time.time()
X_seq_tr, X_static_tr, y_tr = build_sequences(df_train, H)
X_seq_te, X_static_te, y_te = build_sequences(df_test,  H)
print(f'Done in {time.time()-t0:.1f}s')
print(f'Train seq:  {X_seq_tr.shape}  (N, timesteps={H+1}, signals={len(SIGNAL_COLS)})')
print(f'Test seq:   {X_seq_te.shape}')

## Normalization

Unlike tree models, neural networks are **sensitive to input and output scale**.
We apply `StandardScaler` to signals, static params, and targets.
Targets are inverse-transformed before computing the final 3D MAE metric.

In [ ]:
N_tr, T, S = X_seq_tr.shape
N_te       = X_seq_te.shape[0]

sig_scaler    = StandardScaler()
X_seq_tr_norm = sig_scaler.fit_transform(X_seq_tr.reshape(-1,S)).reshape(N_tr,T,S)
X_seq_te_norm = sig_scaler.transform(X_seq_te.reshape(-1,S)).reshape(N_te,T,S)

X_seq_tr_norm = np.nan_to_num(X_seq_tr_norm, nan=0.0)
X_seq_te_norm = np.nan_to_num(X_seq_te_norm, nan=0.0)

static_scaler    = StandardScaler()
X_static_tr_norm = static_scaler.fit_transform(X_static_tr)
X_static_te_norm = static_scaler.transform(X_static_te)

target_scaler = StandardScaler()
y_tr_norm = target_scaler.fit_transform(y_tr)
y_te_norm = target_scaler.transform(y_te)

print('Normalization complete.')
print(f'  Signal mean:   {sig_scaler.mean_.round(2)}')
print(f'  Target scale:  {target_scaler.scale_.round(2)} m')

## PyTorch Dataset & DataLoader

In [ ]:
class LocDataset(Dataset):
    def __init__(self, seq, static, targets):
        # Conv1d expects [batch, channels, length] -> transpose [N,T,S] -> [N,S,T]
        self.seq     = torch.from_numpy(seq.transpose(0,2,1)).float()
        self.static  = torch.from_numpy(static).float()
        self.targets = torch.from_numpy(targets).float()
    def __len__(self):  return len(self.targets)
    def __getitem__(self, i): return self.seq[i], self.static[i], self.targets[i]


train_ds = LocDataset(X_seq_tr_norm, X_static_tr_norm, y_tr_norm)
test_ds  = LocDataset(X_seq_te_norm, X_static_te_norm, y_te_norm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)}  |  Test batches: {len(test_loader)}')
print(f'Seq input: [batch={BATCH_SIZE}, channels={len(SIGNAL_COLS)}, time={H+1}]')

## Model Architecture

**Dual-branch design:**
1. **Temporal branch** — Conv1D blocks with BatchNorm + LeakyReLU + Dropout + Residual skip
2. **Static branch** — FC encoder for device profile (n_antennas, gain, height)
3. **Fusion head** — Dense layers with BatchNorm + Dropout → 3 Cartesian outputs

Weight init: Kaiming (conv layers), Xavier (linear layers)

In [ ]:
class ConvBlock(nn.Module):
    """Conv1d + BatchNorm1d + LeakyReLU + Dropout with residual skip connection."""
    def __init__(self, in_ch, out_ch, kernel=3, dropout=0.2):
        super().__init__()
        self.net  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2),
            nn.BatchNorm1d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(dropout),
        )
        self.skip = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.net(x) + self.skip(x)  # Residual connection


class CNN1DLocator(nn.Module):
    def __init__(self, n_signals, n_static, seq_len,
                 conv_chs=(32,64), dense_units=(128,64), dropout=0.25):
        super().__init__()
        # Temporal branch
        blocks, in_ch = [], n_signals
        for out_ch in conv_chs:
            blocks.append(ConvBlock(in_ch, out_ch, kernel=3, dropout=dropout))
            in_ch = out_ch
        self.conv_branch = nn.Sequential(*blocks)
        self.pool        = nn.AdaptiveAvgPool1d(1)
        # Static branch
        self.static_branch = nn.Sequential(
            nn.Linear(n_static, 16), nn.BatchNorm1d(16), nn.ReLU(inplace=True))
        # Fusion head
        in_dim, head = conv_chs[-1]+16, []
        for out_dim in dense_units:
            head += [nn.Linear(in_dim, out_dim), nn.BatchNorm1d(out_dim),
                     nn.LeakyReLU(0.1, inplace=True), nn.Dropout(dropout)]
            in_dim = out_dim
        head.append(nn.Linear(in_dim, 3))
        self.head = nn.Sequential(*head)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='leaky_relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, seq, static):
        conv_out    = self.pool(self.conv_branch(seq)).squeeze(-1)
        static_out  = self.static_branch(static)
        return self.head(torch.cat([conv_out, static_out], dim=1))


model = CNN1DLocator(
    n_signals=len(SIGNAL_COLS), n_static=len(STATIC_COLS), seq_len=H+1,
    conv_chs=CONV_CHANNELS, dense_units=DENSE_UNITS, dropout=DROPOUT_RATE
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Trainable parameters: {n_params:,}')

## Loss, Optimizer & Scheduler

In [ ]:
if LOSS_FN == 'l1':
    criterion = nn.L1Loss()
    print('Loss: L1 (MAE) — directly optimizes evaluation metric')
elif LOSS_FN == 'mse':
    criterion = nn.MSELoss()
    print('Loss: MSE')
else:
    d_norm    = HUBER_DELTA / target_scaler.scale_.mean()
    criterion = nn.HuberLoss(delta=d_norm)
    print(f'Loss: Huber(delta={HUBER_DELTA}m  -> norm delta={d_norm:.3f})')

# ── Optimizer ─────────────────────────────────────────────────────────
optimizer = optim.Adam(model.parameters(), lr=LR_MIN, weight_decay=WEIGHT_DECAY)

# ── Schedule: Linear Warmup → Cosine Annealing ───────────────────────────
# Phase 1 (epochs 1-WARMUP_EPOCHS): LR ramps linearly from LR_MIN to LR
# Phase 2 (epochs WARMUP_EPOCHS+1 to MAX_EPOCHS): cosine decay from LR to LR_MIN
cosine_epochs = MAX_EPOCHS - WARMUP_EPOCHS

def lr_schedule_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS * (LR / LR_MIN)
    else:
        progress = epoch - WARMUP_EPOCHS
        cos_out = 0.5 * (1.0 + np.cos(np.pi * progress / cosine_epochs))
        desired_lr = LR_MIN + (LR - LR_MIN) * cos_out
        return desired_lr / LR_MIN

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_schedule_lambda)

print(f'Adam(initial_lr={LR_MIN:.0e}, wd={WEIGHT_DECAY})')
print(f'Phase 1 — Linear warmup : {WARMUP_EPOCHS} epochs  {LR_MIN:.0e} → {LR:.0e}')
print(f'Phase 2 — Cosine anneal: {cosine_epochs} epochs  {LR:.0e} → {LR_MIN:.0e}')

## 10B. Learning Rate Finder (Optional / Pre-training)

Implements a **Learning Rate Range Test** (originally proposed by Leslie Smith) to exponentially scale the learning rate and observe the loss behavior. This helps to identify the optimal starting learning rate before launching a full training run.

In [ ]:
class LRFinder:
    def __init__(self, model, optimizer, criterion, device):
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion
        self.device = device
        self.model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        self.optimizer_state = optimizer.state_dict()
        
    def range_test(self, train_loader, start_lr=1e-7, end_lr=10.0, num_iter=100, beta=0.98):
        self.model.load_state_dict({k: v.to(self.device) for k, v in self.model_state.items()})
        self.optimizer.load_state_dict(self.optimizer_state)
        
        lrs, losses = [], []
        mult = (end_lr / start_lr) ** (1.0 / num_iter)
        lr = start_lr
        self.optimizer.param_groups[0]['lr'] = lr
        
        avg_loss = 0.0
        best_loss = float('inf')
        iterator = iter(train_loader)
        
        for iteration in range(1, num_iter + 1):
            try:
                seq, static, tgt = next(iterator)
            except StopIteration:
                iterator = iter(train_loader)
                seq, static, tgt = next(iterator)
                
            seq = seq.to(self.device)
            static = static.to(self.device)
            tgt = tgt.to(self.device)
            
            self.model.train()
            self.optimizer.zero_grad()
            out = self.model(seq, static)
            loss = self.criterion(out, tgt)
            
            avg_loss = beta * avg_loss + (1 - beta) * loss.item()
            smoothed_loss = avg_loss / (1 - beta ** iteration)
            
            if iteration > 1 and smoothed_loss > 4 * best_loss:
                break
                
            if smoothed_loss < best_loss or iteration == 1:
                best_loss = smoothed_loss
                
            losses.append(smoothed_loss)
            lrs.append(lr)
            
            loss.backward()
            self.optimizer.step()
            
            lr *= mult
            self.optimizer.param_groups[0]['lr'] = lr
            
        # Restore states
        self.model.load_state_dict({k: v.to(self.device) for k, v in self.model_state.items()})
        self.optimizer.load_state_dict(self.optimizer_state)
        return lrs, losses


def plot_lr_finder(lrs, losses):
    plt.figure(figsize=(9, 4.5))
    plt.plot(lrs, losses, color='steelblue', lw=2)
    plt.xscale('log')
    plt.xlabel('Learning Rate (log scale)')
    plt.ylabel('Smoothed L1 Loss')
    plt.title('Learning Rate Finder Range Test')
    plt.grid(True, which='both', ls='-', alpha=0.3)
    
    # Approximate the steepest gradient location (skip first 15 steps to avoid init artifacts)
    skip = min(15, len(losses) // 5)
    grads = np.gradient(losses[skip:])
    best_idx = np.argmin(grads) + skip
    best_lr = lrs[best_idx]
    
    plt.axvline(best_lr, color='orangered', ls='--', label=f'Steepest descent: {best_lr:.2e}')
    plt.legend()
    plt.show()
    print(f'Suggested Starting Learning Rate: {best_lr:.2e}')
    return best_lr

### 10C. Run the Learning Rate Finder

Execute this cell to find the optimal learning rate. Note that this test is extremely fast (runs for only 100 mini-batch updates) and restores the model to its original state afterward.

In [ ]:
finder = LRFinder(model, optimizer, criterion, DEVICE)
lrs, losses = finder.range_test(train_loader, start_lr=1e-6, end_lr=1.0, num_iter=100)
suggested_lr = plot_lr_finder(lrs, losses)

# If you want to use the suggested learning rate, update LR in Cell 3 and re-run Cell 10.
# optimizer.param_groups[0]['lr'] = suggested_lr
# print(f'Optimizer learning rate set to: {suggested_lr}')

## Training Loop

In [ ]:
def eval_mae_m(mdl, loader, max_batches=None):
    mdl.eval()
    preds, truths = [], []
    with torch.no_grad():
        for i, (seq, static, tgt) in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            out = mdl(seq.to(DEVICE), static.to(DEVICE)).cpu().numpy()
            preds.append(out); truths.append(tgt.numpy())
    p = target_scaler.inverse_transform(np.vstack(preds))
    t = target_scaler.inverse_transform(np.vstack(truths))
    return float(np.mean(np.linalg.norm(p - t, axis=1)))


def train_epoch(mdl, loader):
    mdl.train()
    total = 0.0
    for seq, static, tgt in loader:
        seq, static, tgt = seq.to(DEVICE), static.to(DEVICE), tgt.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(mdl(seq, static), tgt)
        loss.backward()
        nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item() * seq.size(0)
    return total / len(loader.dataset)

history = {'train_loss': [], 'val_loss': [], 'train_mae_m': [], 'val_mae_m': [], 'lr': []}
best_val, best_mae, best_state, patience_cnt = float('inf'), float('inf'), None, 0

print(f'Training up to {MAX_EPOCHS} epochs | device={DEVICE}')
print(f'Phase 1 — Warmup  : epochs 1-{WARMUP_EPOCHS}  ({LR_MIN:.0e} → {LR:.0e})')
print(f'Phase 2 — Cosine  : epochs {WARMUP_EPOCHS+1}-{MAX_EPOCHS}  ({LR:.0e} → {LR_MIN:.0e})\n')
t_start = time.time()

for epoch in range(1, MAX_EPOCHS+1):
    # ── Advance LR schedule BEFORE training ────────────────────────────
    scheduler.step()
    phase = 'warmup' if epoch <= WARMUP_EPOCHS else 'cosine'

    tr_loss = train_epoch(model, train_loader)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for seq, static, tgt in test_loader:
            v_loss += criterion(model(seq.to(DEVICE), static.to(DEVICE)),
                                tgt.to(DEVICE)).item() * seq.size(0)
    v_loss /= len(test_loader.dataset)
    
    tr_mae  = eval_mae_m(model, train_loader, max_batches=100)
    v_mae   = eval_mae_m(model, test_loader)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(v_loss)
    history['train_mae_m'].append(tr_mae)
    history['val_mae_m'].append(v_mae)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    if v_loss < best_val:
        best_val, best_mae = v_loss, v_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 5 == 0 or epoch == 1 or epoch == MAX_EPOCHS:
        elapsed = time.time() - t_start
        eta     = elapsed / epoch * (MAX_EPOCHS - epoch)
        lr_now  = optimizer.param_groups[0]['lr']
        print(f'Ep {epoch:4d}/{MAX_EPOCHS} | train_loss={tr_loss:.4f} | val_loss={v_loss:.4f} | '
              f'train_MAE={tr_mae:.3f}m | val_MAE={v_mae:.3f}m | lr={lr_now:.2e} ({phase}) | '
              f'patience={patience_cnt}/{EARLY_STOP_PATIENCE} | '
              f'elapsed={elapsed:.0f}s ETA={eta:.0f}s')

    # Only allow early stopping after warmup + a few cosine epochs
    if patience_cnt >= EARLY_STOP_PATIENCE and epoch > WARMUP_EPOCHS + 5:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest 3D MAE: {best_mae:.3f} m  |  total time: {time.time()-t_start:.0f}s')

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val',   color='orangered')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Train / Val Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_mae_m'], color='seagreen', lw=2, label='Val 3D MAE')
axes[1].plot(history['train_mae_m'], color='steelblue', lw=1.5, ls='--', alpha=0.7, label='Train 3D MAE')
axes[1].axhline(7.959, color='gray',      ls='--', lw=1.5, label='XGBoost (7.959m)')
axes[1].axhline(7.506, color='orangered', ls='--', lw=1.5, label='RF h=1 (7.506m)')
axes[1].axhline(best_mae, color='seagreen', ls=':', lw=2, label=f'CNN best ({best_mae:.3f}m)')
axes[1].set(xlabel='Epoch', ylabel='3D MAE (m)', title='Validation 3D MAE vs Baselines')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

axes[2].plot(history['lr'], color='purple', lw=2)
axes[2].set(xlabel='Epoch', ylabel='Learning Rate', title='LR Schedule (Warmup + Cosine)')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {OUT_DIR / "training_curves.png"}')

## Final Evaluation & Comparison

In [ ]:
model.eval()
all_preds, all_truths = [], []
with torch.no_grad():
    for seq, static, tgt in test_loader:
        all_preds.append(model(seq.to(DEVICE), static.to(DEVICE)).cpu().numpy())
        all_truths.append(tgt.numpy())

preds_m  = target_scaler.inverse_transform(np.vstack(all_preds))
truths_m = target_scaler.inverse_transform(np.vstack(all_truths))
errors   = np.linalg.norm(preds_m - truths_m, axis=1)
mae_3d   = float(np.mean(errors))
mpe      = mae_3d / GRID_SPACING
mae_xyz  = np.mean(np.abs(preds_m - truths_m), axis=0)

print(f'CNN  |  3D MAE: {mae_3d:.3f} m  |  MPE: {mpe:.3f} pts')
print(f'     |  X={mae_xyz[0]:.3f}m  Y={mae_xyz[1]:.3f}m  Z={mae_xyz[2]:.3f}m')

comparison = pd.DataFrame([
    {'Model': 'XGBoost (absolute h=3)',    '3D MAE (m)': 7.959, 'MPE': 1.990, 'Features': 19},
    {'Model': 'RF (absolute h=3)',          '3D MAE (m)': 8.090, 'MPE': 2.022, 'Features': 19},
    {'Model': 'RF (hybrid h=1) — best RF', '3D MAE (m)': 7.506, 'MPE': 1.876, 'Features': 11},
    {'Model': f'CNN 1D (h={H}, PyTorch)',   '3D MAE (m)': round(mae_3d,3),
     'MPE': round(mpe,3), 'Features': f'{len(SIGNAL_COLS)}x{H+1}+{len(STATIC_COLS)}'},
]).sort_values('3D MAE (m)')
print('\nComparison Table:')
print(comparison.to_string(index=False))

## Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(errors, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(mae_3d, color='orangered', lw=2, label=f'Mean={mae_3d:.2f}m')
axes[0].axvline(np.median(errors), color='gold', lw=2, ls='--', label=f'Median={np.median(errors):.2f}m')
axes[0].set(xlabel='3D Error (m)', ylabel='Count', title='CNN Error Distribution')
axes[0].legend(); axes[0].grid(alpha=0.3)

se  = np.sort(errors)
cdf = np.arange(1, len(se)+1) / len(se)
axes[1].plot(se, cdf*100, color='steelblue', lw=2, label='CNN 1D')
axes[1].axvline(7.959, color='gray',  ls=':', lw=1.5, label='XGBoost (7.959m)')
axes[1].axvline(7.506, color='black', ls=':', lw=1.5, label='RF h=1 Hybrid (7.506m)')
axes[1].axvline(mae_3d, color='orangered', ls='--', lw=2, label=f'CNN ({mae_3d:.2f}m)')
axes[1].set(xlabel='3D Error (m)', ylabel='CDF (%)', title='Cumulative Error Distribution')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'error_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Model & Scalers

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'hyperparams': {
        'H': H, 'conv_chs': CONV_CHANNELS, 'dense_units': DENSE_UNITS,
        'dropout': DROPOUT_RATE, 'loss_fn': LOSS_FN,
    },
    'results': {'mae_3d_m': mae_3d, 'mpe': mpe, 'mae_xyz': mae_xyz.tolist()},
}, OUT_DIR / 'cnn_model.pt')

joblib.dump(sig_scaler,    OUT_DIR / 'sig_scaler.pkl')
joblib.dump(static_scaler, OUT_DIR / 'static_scaler.pkl')
joblib.dump(target_scaler, OUT_DIR / 'target_scaler.pkl')

print(f'Model + scalers saved to: {OUT_DIR}')
print(f'Final: 3D MAE = {mae_3d:.3f} m  |  MPE = {mpe:.3f} pts')

## Generate Run Report

In [ ]:
import datetime

def save_report(hist, final_mae, final_mpe, final_xyz, n_params, time_sec):
    t_stamp = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
    report_p = OUT_DIR / f'cnn_run_report_{t_stamp}.md'
    
    # Build markdown table of logs
    table_lines = [
        '| Epoch | Train Loss | Val Loss | Train MAE (m) | Val MAE (m) | LR |',
        '| :---: | :---: | :---: | :---: | :---: | :---: |'
    ]
    for i in range(len(hist['train_loss'])):
        tr_l  = hist['train_loss'][i]
        v_l   = hist['val_loss'][i]
        tr_m  = hist['train_mae_m'][i]
        val_m = hist['val_mae_m'][i]
        lr    = hist['lr'][i]
        table_lines.append(f'| {i+1} | {tr_l:.4f} | {v_l:.4f} | {tr_m:.3f} | {val_m:.3f} | {lr:.1e} |')
    
    table_content = '\n'.join(table_lines)
    
    report_content = f"""# 1D-CNN Localization Exploration Report

* **Timestamp:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
* **Run ID:** `cnn_run_report_{t_stamp}`

## 1. Experimental Configuration

* **History Depth (h):** {H} (sequence length = {H+1} timesteps)
* **Training Subsample:** {SUBSAMPLE * 100:.1f}%
* **Loss Function:** `{LOSS_FN.upper()}`
* **Learning Rate:** {LR} peak, warmup {WARMUP_EPOCHS} epochs, cosine anneal to {LR_MIN}
* **Batch Size:** {BATCH_SIZE}
* **Weight Decay:** {WEIGHT_DECAY}
* **Model Architecture Detail:**
  - Conv Channels: `{CONV_CHANNELS}`
  - Dense Head Units: `{DENSE_UNITS}`
  - Dropout Rate: `{DROPOUT_RATE}`
* **Model Parameters:** {n_params:,} trainable parameters
* **Total Training Time:** {time_sec:.1f} seconds

## 2. Final Results Summary

| Metric | CNN 1D Exploration | RF h=1 Hybrid Baseline | XGBoost h=3 Absolute Baseline |
| :--- | :---: | :---: | :---: |
| **3D Position MAE** | **{final_mae:.3f} m** | 7.506 m | 7.959 m |
| **Mean Position Error (MPE)** | **{final_mpe:.3f} pts** | 1.876 pts | 1.990 pts |
| **X MAE** | **{final_xyz[0]:.3f} m** | — | — |
| **Y MAE** | **{final_xyz[1]:.3f} m** | — | — |
| **Z MAE** | **{final_xyz[2]:.3f} m** | — | — |

## 3. Epoch Training Log

{table_content}

## 4. Key Findings and Physical Analysis

1. **Comparison with Tree-based Baselines:**
   - The 1D-CNN with L1 Loss achieved a best 3D MAE of **{final_mae:.3f} m**, """
    f"""which {'beats' if final_mae < 7.506 else 'lags behind'} our best Random Forest baseline (**7.506 m**).

## 5. Recommended Next Steps
- Review the training curves for signs of underfitting or overfitting.
- Vary Sequence Length (h): Test h=5 or h=10 for longer temporal context.
- Experiment with Huber loss if large outliers dominate the error distribution.
"""
    report_content = report_content.replace('{table_content}', table_content)
    report_p.write_text(report_content, encoding='utf-8')
    print(f'Report written to: {report_p}')

save_report(history, mae_3d, mpe, mae_xyz, n_params, time.time() - t_start)
